# Паттерн 13: Ensemble (Cascade) — каскадная стратегия

Ensemble обобщает идею Voting. Самая практичная стратегия — каскад: быстрая дешёвая модель обрабатывает запрос первой, и только если её уверенность ниже порога, подключается мощная дорогая модель. На практике 80% простых запросов решаются одним дешёвым вызовом, и суммарная стоимость снижается в разы.

Никаких циклов — граф линейный с одной развилкой. Простые запросы проходят по короткому пути (fast -> accept -> END), сложные — по длинному (fast -> strong -> END).

```mermaid
graph LR
    START((START)) --> fast_model(["🔹 Fast Model<br/><i>генерирует вариант</i>"])
    fast_model -->|high confidence| accept_fast(["🔹 Accept Fast<br/><i>принимает ответ</i>"])
    fast_model -->|low confidence| strong_model(["🔹 Strong Model<br/><i>генерирует развёрнутый ответ</i>"])
    accept_fast --> END((END))
    strong_model --> END

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px

    class START,END terminal
    class fast_model,accept_fast,strong_model worker
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


In [3]:
llm = get_llm()

## Состояние каскада

Состояние `CascadeState` хранит вопрос пользователя, ответ быстрой модели и её самооценку уверенности (`HIGH` или `LOW`), ответ мощной модели (если понадобился), итоговый ответ и флаг эскалации. В реальной системе `fast_llm` — дешёвая модель (Haiku), `strong_llm` — мощная (Opus). Здесь разницу моделируем промптами.

In [4]:
class CascadeState(TypedDict):
    question: str
    fast_answer: str
    fast_confidence: str
    strong_answer: str
    final_answer: str
    used_strong: bool

## Быстрая модель с самооценкой

Быстрый агент отвечает коротко (1-2 предложения) и в последней строке указывает уровень уверенности: `CONFIDENCE: HIGH` или `CONFIDENCE: LOW`. Мы извлекаем эту метку из ответа и сохраняем отдельно в поле `fast_confidence`. Именно это поле определит, пойдёт ли запрос дальше к мощной модели.

In [5]:
def fast_agent(state: CascadeState) -> dict:
    """Быстрая модель: пытается ответить и оценивает свою уверенность."""
    response = llm.invoke([
        SystemMessage(content="""Ты — быстрый ассистент. Ответь коротко (1-2 предложения).

В ПОСЛЕДНЕЙ строке ответа напиши уровень уверенности:
CONFIDENCE: HIGH — если ты уверен в ответе
CONFIDENCE: LOW — если вопрос сложный или неоднозначный"""),
        HumanMessage(content=state["question"])
    ])

    content = response.content.strip()
    lines = content.split("\n")

    # Извлекаем confidence из последней строки
    confidence = "LOW"
    if "CONFIDENCE:" in lines[-1].upper():
        confidence = "HIGH" if "HIGH" in lines[-1].upper() else "LOW"
        answer = "\n".join(lines[:-1]).strip()
    else:
        answer = content

    print(f"  [Fast] Ответ: {answer[:80]}...")
    print(f"  [Fast] Уверенность: {confidence}")
    return {"fast_answer": answer, "fast_confidence": confidence}

## Маршрутизация и обработка результатов

Функция `should_escalate` — это условное ребро графа. Если быстрая модель уверена (`HIGH`), ответ принимается как есть через узел `accept_fast`. Если не уверена (`LOW`) — запрос эскалируется к мощной модели `strong_agent`, которая получает и оригинальный вопрос, и предварительный ответ быстрой модели для контекста.

In [6]:
def should_escalate(state: CascadeState) -> str:
    """Маршрутизация по уверенности быстрой модели."""
    if state["fast_confidence"] == "HIGH":
        return "accept"
    return "escalate"


def accept_fast(state: CascadeState) -> dict:
    """Принимаем ответ быстрой модели — эскалация не нужна."""
    print(f"  [Cascade] Быстрый ответ принят (без эскалации)")
    return {"final_answer": state["fast_answer"], "used_strong": False}


def strong_agent(state: CascadeState) -> dict:
    """Мощная модель: подключается, если быстрая не уверена."""
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — экспертный аналитик. Дай развёрнутый, "
            "детальный ответ (4-5 предложений). Рассмотри "
            "вопрос с разных сторон. Будь точен и обоснован."
        )),
        HumanMessage(content=(
            f"Вопрос: {state['question']}\n\n"
            f"Предварительный ответ быстрого агента (возможно неточный):\n"
            f"{state['fast_answer']}"
        ))
    ])
    print(f"  [Strong] Развёрнутый ответ готов ({len(response.content)} символов)")
    return {
        "strong_answer": response.content,
        "final_answer": response.content,
        "used_strong": True,
    }

## Сборка графа с двумя путями

Граф содержит три узла: `fast`, `accept`, `strong`. Из `fast` идёт условное ребро с двумя ветками: при высокой уверенности — к `accept`, при низкой — к `strong`. Оба пути завершаются в `END`. Это линейный граф без циклов: каждый запрос проходит ровно два узла.

In [7]:
graph = StateGraph(CascadeState)

graph.add_node("fast", fast_agent)
graph.add_node("accept", accept_fast)
graph.add_node("strong", strong_agent)

graph.add_edge(START, "fast")
graph.add_conditional_edges("fast", should_escalate, {
    "accept": "accept",
    "escalate": "strong",
})
graph.add_edge("accept", END)
graph.add_edge("strong", END)

app = graph.compile()

## Запуск

Тестируем каскад на трёх вопросах разной сложности. Простые фактологические вопросы ("2 + 2", "столица Франции") должны решаться быстрой моделью с высокой уверенностью. Сложный аналитический вопрос про архитектурные компромиссы должен вызвать эскалацию к мощной модели.

In [8]:
questions = [
    "Сколько будет 2 + 2?",
    "Какие архитектурные компромиссы возникают при переходе от монолитного агента к мультиагентной системе?",
    "Столица Франции?",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"  Вопрос: {q}")
    print(f"{'=' * 60}")
    result = app.invoke({
        "question": q,
        "fast_answer": "", "fast_confidence": "",
        "strong_answer": "", "final_answer": "",
        "used_strong": False,
    })
    level = "Strong (эскалация)" if result["used_strong"] else "Fast (без эскалации)"
    print(f"  Уровень: {level}")
    print(f"  Ответ: {result['final_answer'][:200]}...")


  Вопрос: Сколько будет 2 + 2?


  [Fast] Ответ: 2 + 2 = 4....
  [Fast] Уверенность: HIGH
  [Cascade] Быстрый ответ принят (без эскалации)
  Уровень: Fast (без эскалации)
  Ответ: 2 + 2 = 4....

  Вопрос: Какие архитектурные компромиссы возникают при переходе от монолитного агента к мультиагентной системе?


  [Fast] Ответ: При переходе от монолита к мультиагентной системе обычно меняют простоту и предс...
  [Fast] Уверенность: HIGH
  [Cascade] Быстрый ответ принят (без эскалации)
  Уровень: Fast (без эскалации)
  Ответ: При переходе от монолита к мультиагентной системе обычно меняют простоту и предсказуемость на модульность, масштабируемость и специализацию: появляется сложнее координация, больше накладных расходов н...

  Вопрос: Столица Франции?


  [Fast] Ответ: Столица Франции — Париж....
  [Fast] Уверенность: HIGH
  [Cascade] Быстрый ответ принят (без эскалации)
  Уровень: Fast (без эскалации)
  Ответ: Столица Франции — Париж....


## Итог

Каскадная стратегия Ensemble — один из самых практичных паттернов оптимизации стоимости. Ключевые идеи:

- **Самооценка уверенности** — быстрая модель сама решает, достаточно ли хорош её ответ, через метку `CONFIDENCE: HIGH/LOW` в конце ответа.
- **Условная эскалация** — мощная модель подключается только при `LOW`, экономя ресурсы на простых запросах.
- **Линейный граф без циклов** — всего одна развилка, два возможных пути, максимум два вызова LLM на запрос.
- **В продакшене** — вместо промптов используются разные модели (`claude-haiku-4-5` для fast, `claude-sonnet-4-6` для strong), что даёт реальную экономию в разы.